# gdpr rag: bi-encoder vs cross-encoder rerank

two retrieval pipelines over the same gdpr chunks, same llm, same prompt

- pipeline 1 (baseline): bi-encoder top 10 → top 3 → llm → answer 1
- pipeline 2 (rerank): same top 10 → cross-encoder rescores all 10 → top 3 → llm → answer 2

run top to bottom. needs the venv from `requirements.txt` and ollama running with `gemma2:9b-instruct-q4_K_M` pulled.
`data/` = inputs (pdf, hand labels), `results/` = everything this notebook writes

## setup

In [1]:
#quiet the windows / hf noise before anything gets imported
import logging
import os
import warnings

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="IProgress not found")  # no ipywidgets, plain text progress is fine

import hashlib
import importlib.metadata
import json
import platform
import random
import re
import subprocess
import time
import urllib.request
from datetime import datetime
from pathlib import Path

import faiss
import numpy as np
import ollama
import pandas as pd
import pymupdf
import torch
import transformers
import yaml
from huggingface_hub import try_to_load_from_cache
from huggingface_hub.utils import logging as hf_logging
from sentence_transformers import CrossEncoder, SentenceTransformer

transformers.logging.set_verbosity_error()
hf_logging.set_verbosity_error()  # hides the "unauthenticated requests" nag, models are public
transformers.utils.logging.disable_progress_bar()
pd.set_option("display.max_colwidth", 120)

In [2]:
#config, everything tweakable lives here
ROOT = Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

BI_ENCODER = "sentence-transformers/msmarco-MiniLM-L6-cos-v5"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L6-v2"  # spec says ms-marco-MiniLM-L-6-v2, hf redirects to this
LLM = "gemma2:9b-instruct-q4_K_M"

TOP_K = 10  # candidates from the bi-encoder
TOP_N = 3  # chunks that go into the prompt
MAX_CHUNK_TOKENS = 350  # bi-encoder cuts at 384, leave room
LLM_OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 4096, "num_predict": 400}
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

device: cuda NVIDIA GeForce RTX 4070 Laptop GPU


In [3]:
#both encoders, same minilm-l6 size, both trained on ms marco
bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE, max_length=512)
tokenizer = bi_encoder.tokenizer  # all chunk token counts use this one

print("bi-encoder:", BI_ENCODER, "| max_seq_length", bi_encoder.max_seq_length, "| dim", bi_encoder.get_embedding_dimension())
print("bi-encoder modules:", [type(module).__name__ for module in bi_encoder])  # ends in Normalize -> dot = cosine
print("cross-encoder:", CROSS_ENCODER, "| max_seq_length", cross_encoder.max_seq_length, "| activation", cross_encoder.activation_fn)  # Identity -> raw logits

bi-encoder: sentence-transformers/msmarco-MiniLM-L6-cos-v5 | max_seq_length 384 | dim 384
bi-encoder modules: ['Transformer', 'Pooling', 'Normalize']
cross-encoder: cross-encoder/ms-marco-MiniLM-L6-v2 | max_seq_length 512 | activation Identity()


In [4]:
#ollama has to be up with the model pulled, first call also loads it into memory
def ollama_placement(name: str) -> str:
    #ollama ps shows how much of the model sits on gpu vs cpu
    listing = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
    row = next((line for line in listing.splitlines() if line.startswith(name)), "")
    found = re.search(r"\d+%/\d+% CPU/GPU|\d+% (?:GPU|CPU)", row)
    return found.group() if found else "not loaded"


reply = ollama.chat(
    model=LLM,
    messages=[{"role": "user", "content": "reply with just: ok"}],
    options={**LLM_OPTIONS, "num_predict": 5},
    keep_alive="30m",
)
print("ollama says:", reply.message.content.strip(), "| placement:", ollama_placement(LLM))

ollama says: ok | placement: 24%/76% CPU/GPU


In [5]:
#run info, so every number here can be traced back to exact versions + hardware
def hf_revision(repo_id: str) -> str:
    return Path(try_to_load_from_cache(repo_id, "config.json")).parent.name  # .../snapshots/<commit>/config.json


def command_output(command: list) -> str:
    return subprocess.run(command, capture_output=True, text=True).stdout.strip()


PACKAGES = ("torch", "sentence-transformers", "transformers", "faiss-cpu", "pymupdf", "ollama", "numpy", "pandas")
RUN_INFO = {
    "ran_at": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "os": platform.platform(),
    "gpu": command_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]),
    "cuda": torch.version.cuda,
    "device": DEVICE,
    "packages": {name: importlib.metadata.version(name) for name in PACKAGES},
    "bi_encoder": {"id": BI_ENCODER, "revision": hf_revision(BI_ENCODER)},
    "cross_encoder": {"id": CROSS_ENCODER, "revision": hf_revision(CROSS_ENCODER)},
    "llm": {
        "id": LLM,
        "digest": next(model.digest for model in ollama.list().models if model.model == LLM),
        "ollama": re.search(r"\d+\.\d+\.\d+", command_output(["ollama", "--version"])).group(),
        "placement": ollama_placement(LLM),
        "options": LLM_OPTIONS,
    },
}


def save_run_info() -> None:
    (RESULTS / "run_info.json").write_text(json.dumps(RUN_INFO, indent=2), encoding="utf-8")


save_run_info()
print(json.dumps(RUN_INFO, indent=2))

{
  "ran_at": "2026-10-17T20:07:08",
  "python": "3.11.9",
  "os": "Windows-10-10.0.26300-SP0",
  "gpu": "NVIDIA GeForce RTX 4070 Laptop GPU, 8188 MiB, 617.42",
  "cuda": "12.8",
  "device": "cuda",
  "packages": {
    "torch": "2.11.0+cu128",
    "sentence-transformers": "6.1.0",
    "transformers": "5.20.1",
    "faiss-cpu": "1.15.1",
    "pymupdf": "1.28.2",
    "ollama": "0.6.3",
    "numpy": "2.4.6",
    "pandas": "3.0.6"
  },
  "bi_encoder": {
    "id": "sentence-transformers/msmarco-MiniLM-L6-cos-v5",
    "revision": "14ca9be4bbcf1402eac0f43a2e2ccb6e0f994ba3"
  },
  "cross_encoder": {
    "id": "cross-encoder/ms-marco-MiniLM-L6-v2",
    "revision": "233902d25c440f23af6f7d6e94d2946bac0bee0a"
  },
  "llm": {
    "id": "gemma2:9b-instruct-q4_K_M",
    "digest": "c20bec88025f3aefa420efd64dca0ee4abc8e36078d6b74b0f660cdcbd5d1982",
    "ollama": "0.34.2",
    "placement": "24%/76% CPU/GPU",
    "options": {
      "temperature": 0,
      "seed": 42,
      "num_ctx": 4096,
      "num_pre